# FinTok AI · 04_video / 02 · Avatar realista con EchoMimic (Colab)

**Objetivo:** generar el vídeo de la cara hablando de cada escena con **EchoMimic** (versión acelerada) en la GPU gratuita de Colab, y montar el vídeo final.

**Problema técnico:** EchoMimic necesita Python 3.10 y versiones antiguas de PyTorch/diffusers; Colab usa Python 3.13.
**Solución:** un entorno aparte con Python 3.10 creado con `uv`, solo para EchoMimic.

**Entrada:** el JSON de escenas, los avatares de cada `speaker` (`<speaker>_closed.png`, `_half.png`, `_open.png`) y `compositor.py`.
**Salida:** un vídeo por escena en `outputs/avatar_videos/`, el JSON con `avatar_video_path` y el vídeo final.

> Alternativa descartada: llamar al Space público de Hugging Face (`fffiloni/EchoMimic`). Funcionaba, pero la cuota gratuita de ZeroGPU (≈200 s por escena) solo daba para una escena al día.

⚠️ Necesita **GPU T4** (Entorno de ejecución → Cambiar tipo de entorno).

## 0. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Entorno Python 3.10 para EchoMimic (~5 min)

In [ ]:
%%writefile /content/echomimic_requirements.txt
torch==2.2.2
torchvision==0.17.2
torchaudio==2.2.2
numpy<2
diffusers==0.24.0
huggingface_hub==0.25.2
transformers==4.38.2
accelerate==0.27.2
omegaconf==2.3.0
einops==0.4.1
av>=12,<14
moviepy==1.0.3
ffmpeg-python==0.2.0
opencv-python-headless<4.11
facenet_pytorch==2.5.0
more-itertools
regex
tqdm
safetensors

In [ ]:
# Colab fuerza a uv a usar su Python 3.13 (UV_SYSTEM_PYTHON); lo desactivamos solo para estos comandos
!pip install -q uv
!uv python install 3.10
!env -u UV_SYSTEM_PYTHON uv venv --clear --python 3.10 --python-preference only-managed /content/emenv
!env -u UV_SYSTEM_PYTHON uv pip install -q --python /content/emenv/bin/python -r /content/echomimic_requirements.txt
!/content/emenv/bin/python -c "import torch, diffusers; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| diffusers', diffusers.__version__)"

## 2. Código y pesos del modelo (~10 GB)

Solo los pesos de la versión acelerada audio→vídeo (el repositorio completo pesa más de 60 GB).

In [ ]:
!git clone -q https://github.com/BadToBest/EchoMimic /content/EchoMimic

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    "BadToBest/EchoMimic",
    local_dir="/content/EchoMimic/pretrained_weights",
    allow_patterns=["sd-image-variations-diffusers/unet/*", "sd-vae-ft-mse/*",
                    "audio_processor/whisper_tiny.pt", "denoising_unet_acc.pth",
                    "reference_unet.pth", "face_locator.pth", "motion_module_acc.pth"],
)
!du -sh /content/EchoMimic/pretrained_weights

## 3. Subir archivos

Sube a la vez: el JSON de escenas, `compositor.py` y los PNG de los avatares de todas las personas que hablan (por ejemplo `andrea_*.png` y `miriam_*.png`).
Puedes ejecutar la celda varias veces si están en carpetas distintas.

In [ ]:
import os, json, glob, shutil
from google.colab import files
from PIL import Image

os.makedirs("assets/avatar", exist_ok=True)
for name in files.upload():
    clean = name.replace(" (1)", "").replace(" (2)", "")   # Colab renombra si ya existía
    if clean.endswith(".png"):
        shutil.move(name, f"assets/avatar/{clean}")
    elif name != clean:
        os.replace(name, clean)

print("Avatares:", sorted(os.listdir("assets/avatar")))
print("JSON:", [f for f in os.listdir(".") if f.endswith(".json")])

In [ ]:
JSON_PATH = "aapl_2024.json"   # cambia por el nombre de tu JSON
DEFAULT_SPEAKER = "andrea"     # si una escena no indica quién habla

spec = json.load(open(JSON_PATH, encoding="utf-8"))
for s in spec["scenes"]:
    s["speaker"] = (s.get("speaker") or DEFAULT_SPEAKER).strip().lower()  # "ANDREA" → "andrea"
speakers = sorted({s["speaker"] for s in spec["scenes"]})
faltan = [sp for sp in speakers if not os.path.exists(f"assets/avatar/{sp}_closed.png")]
print(spec["company"], spec["year"], "·", len(spec["scenes"]), "escenas · hablan:", speakers)
assert not faltan, f"Faltan avatares de: {faltan}"

## 4. Preparar las entradas

- **Avatares:** EchoMimic no acepta transparencia; ponemos cada avatar sobre el mismo fondo liso que la burbuja del montador.
- **Audio:**
  - Si la escena trae `audio_path` (la voz real de la parte de audio, en cualquier formato), la convertimos a WAV mono 16 kHz, que es lo que espera EchoMimic.
  - Si no, generamos una voz de prueba con gTTS, con un **acento distinto por persona** para que se distingan las dos voces.

In [ ]:
!pip install -q gTTS
from gtts import gTTS

os.makedirs("inputs", exist_ok=True)
FLAT = {}
for sp in speakers:
    img = Image.open(f"assets/avatar/{sp}_closed.png").convert("RGBA")
    base = Image.new("RGBA", img.size, (225, 233, 245, 255))
    base.alpha_composite(img)
    FLAT[sp] = os.path.abspath(f"inputs/{sp}_flat.png")
    base.convert("RGB").resize((512, 512), Image.LANCZOS).save(FLAT[sp])

# voz de prueba distinta para cada persona (acento de Google TTS)
VOCES = {"andrea": "com", "miriam": "co.uk"}      # EE. UU. / Reino Unido

for s in spec["scenes"]:
    wav = os.path.abspath(f"inputs/{spec['company']}_{s['scene_id']:02d}.wav")
    if s.get("audio_path") and os.path.exists(s["audio_path"]):
        src_audio = s["audio_path"]                                   # voz real
    else:
        src_audio = wav.replace(".wav", ".mp3")                       # voz de prueba
        gTTS(s["text"], lang=spec.get("language", "es"), tld=VOCES.get(s["speaker"], "com")).save(src_audio)
    if os.path.abspath(src_audio) != wav:
        os.system(f'ffmpeg -y -loglevel error -i "{src_audio}" -ar 16000 -ac 1 "{wav}"')
    s["audio_path"] = wav
print({s["scene_id"]: (s["speaker"], os.path.basename(s["audio_path"])) for s in spec["scenes"]})

## 5. Generar los vídeos con EchoMimic

Cada escena se genera con el avatar de quien habla. Empieza con `SOLO_PRUEBA = True` (solo la escena más corta).
Las escenas que ya tienen vídeo no se repiten.

In [ ]:
SOLO_PRUEBA = True

escenas = [s for s in spec["scenes"] if s.get("verified", True) and not s.get("avatar_video_path")]
if SOLO_PRUEBA and escenas:
    escenas = [min(escenas, key=lambda s: len(s["text"]))]

cfg = open("/content/EchoMimic/configs/prompts/animation_acc.yaml").read().split("## test cases")[0]
cfg += "## test cases\ntest_cases:\n"
for sp in speakers:
    mias = [s for s in escenas if s["speaker"] == sp]
    if mias:
        cfg += f'  "{FLAT[sp]}":\n' + "".join(f'    - "{s["audio_path"]}"\n' for s in mias)
open("/content/EchoMimic/configs/prompts/fintok.yaml", "w").write(cfg)
print(cfg.split("## test cases")[1])

In [ ]:
%cd /content/EchoMimic
!/content/emenv/bin/python -u infer_audio2vid_acc.py --config ./configs/prompts/fintok.yaml
%cd /content

## 6. Asociar cada vídeo a su escena

In [ ]:
os.makedirs("outputs/avatar_videos", exist_ok=True)
for s in spec["scenes"]:
    stem = os.path.splitext(os.path.basename(s["audio_path"]))[0]
    hits = sorted(glob.glob(f"/content/EchoMimic/output/**/{s['speaker']}_flat_{stem}_*withaudio.mp4", recursive=True))
    if hits:
        dst = f"outputs/avatar_videos/{spec['company']}_{spec['year']}_scene{s['scene_id']:02d}.mp4"
        shutil.copy(hits[-1], dst)
        s["avatar_video_path"] = dst
    print(("✓" if s.get("avatar_video_path") else "·"), "escena", s["scene_id"], s["speaker"], s.get("avatar_video_path", "(avatar de respaldo)"))

with open("outputs/spec_with_avatar.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, ensure_ascii=False, indent=2)

Si te ha gustado la prueba, vuelve a la sección 5 con `SOLO_PRUEBA = False` y repite 5 y 6.

## 7. Montar el vídeo final

Si has reiniciado la sesión, la primera línea recupera el JSON guardado en la sección 6.

In [ ]:
from IPython.display import Video
from compositor import render_video

if os.path.exists("outputs/spec_with_avatar.json"):
    spec = json.load(open("outputs/spec_with_avatar.json", encoding="utf-8"))
out = render_video(spec, avatar_dir="assets/avatar",
                   out_path=f"outputs/{spec['company']}_{spec['year']}_final.mp4")
Video(out, embed=True, width=360)

## 8. Descargar resultados

In [ ]:
!zip -qr echomimic_resultados.zip outputs
files.download("echomimic_resultados.zip")